# SwarKavach: run the whole thing

One notebook, start to finish. It fetches the real corpora, builds the Hindi
anti-spoofing set, generates the Hinglish call corpus with neural voices,
trains every model, evaluates, and hands you a zip to drop back into the repo.

**Runtime**: CPU is fine. Nothing in this notebook uses a GPU, so leave the
runtime on CPU and keep your GPU quota for notebook 02, which does need one.

**Time**: 75 to 165 minutes, honestly. Install 2, tests 2, download 5, pairs
10, corpus 25 to 60, train 10 to 20, evaluate 25 to 60, Drive uploads 5.

**If you get disconnected**: nothing is lost. Every expensive artifact is
checkpointed to your Drive as it is produced and pulled back at the start of
the next run. Re-open, run from the top, and it skips what it already has.

**These cells are deliberately thin.** Each one calls a function in
`swarkavach.runner` inside the clone. A Colab notebook runs from wherever you
opened it, not from the clone, so `git pull` updates the code and cannot
update the cells. Keeping the logic in the repo means a fix reaches you
through a pull. If the cells themselves ever have to change, the cell below
says so instead of letting you find out from a strange error.


## 0. Clone and install

### Your GitHub token

The repo can be private. Put a Personal Access Token in Colab's **Secrets**
panel (the key icon in the left sidebar) under the name `GH_PAT`, and switch
on "Notebook access". A fine-grained token needs **Contents: read and write**
on this one repository, nothing else.

Secrets are the right place for it. Pasting a token into a cell puts it in the
notebook file, which then travels with any copy you share. Nothing below ever
prints it, and the token is stripped from the git remote right after cloning
so it is not left sitting in `.git/config`.

One consequence worth knowing: `!git pull` on its own fails with
`could not read Username for 'https://github.com'`, because there is no
credential left in the clone for it to use. To update, re-run the clone cell
below rather than pulling by hand. It re-authenticates for one fetch and
cleans up after itself.


In [ ]:
# CHANGE THIS to your own repo.
REPO_SLUG = "BhavyeNijhawan/svarkavach"     # user/repo
BRANCH    = "main"

import os, subprocess, shutil, getpass
from pathlib import Path

# Deliberately NOT "swarkavach". Colab keeps /content on sys.path, so a
# directory there whose name matches the package shadows the real install
# as an empty namespace package, and you get "cannot import name config
# from swarkavach (unknown location)" much later on. A different directory
# name avoids the collision entirely.
CLONE_DIR = "/content/svk_repo" if Path("/content").is_dir() else "svk_repo"

TOKEN = ""
try:
    from google.colab import userdata
    TOKEN = userdata.get("GH_PAT") or ""
except Exception:
    pass
if not TOKEN:
    print("No GH_PAT secret found. Leave blank if the repo is public.")
    TOKEN = getpass.getpass("GitHub token (hidden, optional): ").strip()

# never build a URL with the token into a printable variable
def _auth_url():
    return (f"https://{TOKEN}@github.com/{REPO_SLUG}.git" if TOKEN
            else f"https://github.com/{REPO_SLUG}.git")

CLEAN_URL = f"https://github.com/{REPO_SLUG}.git"
print("repo:", CLEAN_URL, "| token:", "yes" if TOKEN else "no")


In [ ]:
# Re-run THIS cell to update the clone. A bare `!git pull` in a cell of your
# own will fail with "could not read Username for https://github.com", because
# the token is deliberately stripped from .git/config at the bottom of this
# cell. That is the point: a private repo's credential should not sit in a
# file inside a runtime other people can be handed. This cell puts it back for
# the length of one fetch and removes it again.
if Path(f"{CLONE_DIR}/.git").exists():
    print("already cloned, updating")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", _auth_url()],
                   check=False, capture_output=True)

    def _head():
        return subprocess.run(["git", "-C", CLONE_DIR, "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()

    before = _head()
    r = subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", BRANCH],
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or "") + (r.stdout or "")
        if TOKEN:
            msg = msg.replace(TOKEN, "***")
        subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", CLEAN_URL],
                       check=False, capture_output=True)
        print(f"pull failed, still on {before}")
        print(msg[-700:])
        print()
        print("Most often a run wrote inside the clone, or Colab autosaved a")
        print("notebook. Nothing here is worth keeping, so take the remote copy:")
        print(f"  !git -C {CLONE_DIR} fetch origin {BRANCH} && \\")
        print(f"   git -C {CLONE_DIR} reset --hard origin/{BRANCH}")
        print("then re-run this cell.")
        # Loud on purpose. Silently continuing on stale code means the next
        # hour of synthesis and training tests the wrong commit.
        raise SystemExit("not on the latest code, stopping")
    after = _head()
    print(f"updated {before} -> {after}" if before != after else f"already current at {after}")
else:
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    r = subprocess.run(["git", "clone", "--branch", BRANCH, _auth_url(), CLONE_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # scrub the token out of anything we show
        msg = (r.stderr or "").replace(TOKEN, "***") if TOKEN else (r.stderr or "")
        raise SystemExit("clone failed: " + msg[-800:])
    print("cloned")

# strip the token back out of the stored remote
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", CLEAN_URL],
               check=False, capture_output=True)

os.chdir(CLONE_DIR)
print("cwd:", os.getcwd())
_r = subprocess.run(["git", "remote", "-v"], capture_output=True, text=True).stdout.splitlines()
print("remote:", _r[0] if _r else "none")
_log = subprocess.run(["git", "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("at:", _log)


In [ ]:
# Install. Colab already has torch, numpy and friends, so this is mostly small
# packages plus an editable install of the project itself.
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip install -q edge-tts indic-transliteration soundfile 2>&1 | tail -3
!pip install -q -e . 2>&1 | tail -3

# Verify rather than assume. A quiet pip failure here shows up much later as a
# confusing ImportError, so check it now and say exactly what went wrong.
import importlib, sys
from pathlib import Path

src = Path.cwd() / "src"
if src.is_dir() and str(src) not in sys.path:
    sys.path.insert(0, str(src))      # belt and braces if the editable install failed

for m in list(sys.modules):
    if m == "swarkavach" or m.startswith("swarkavach."):
        del sys.modules[m]

try:
    import swarkavach
    from swarkavach import config
    print("swarkavach imports from:", swarkavach.__file__)
    print("version:", swarkavach.__version__)
except ImportError as exc:
    print("swarkavach did not import:", exc)
    print("cwd:", Path.cwd(), "| src exists:", src.is_dir())
    print("Re-run the clone cell, then this one.")
    raise


In [ ]:
# The cells below are thin wrappers over swarkavach.runner, which lives in
# the clone. NB_VERSION says which cell layout this notebook is; the runner
# checks it against what the code expects and tells you to re-open the
# notebook if they have diverged. Everything else reaches you via a pull.
NB_VERSION = 1

from swarkavach import runner

if not runner.check_notebook(NB_VERSION):
    raise SystemExit("re-open the notebook from GitHub, then run again")

# runner.preflight(full=True) adds the integration test, which generates a
# corpus and trains on it. That took 37 minutes on Colab and the stages
# below do the same work for real, so it is off by default.
runner.preflight()


In [ ]:
# Environment check. Nothing below needs a GPU except the optional RawNet and
# MuRIL steps, but it is worth knowing what you have.
import shutil, sys, torch
from swarkavach import config

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))
try:
    import psutil
    print("ram         : %.1f GB" % (psutil.virtual_memory().total / 1e9))
except Exception:
    pass
print("disk free   : %.1f GB" % (shutil.disk_usage('.').free / 1e9))
print("ffmpeg      :", "yes" if config.find_ffmpeg() else "no (G.711 still exact, in numpy)")


### Drive checkpoints

Colab throws the local disk away when the session ends, and free sessions end
often. Every expensive artifact is copied to your Drive as it is produced and
pulled back at the start of the next run, so a disconnect costs you the stage
you were in the middle of and nothing before it.

The one that actually matters is the speech cache. Building it is thousands of
rate-limited synthesis requests and takes about an hour. Building it twice is
an hour you do not get back.

A stage is skipped only when Drive actually handed its output back in this
session. Not when files happen to exist: the git checkout ships files under
`data/`, and counting those told a brand new session that evaluation was
already done, so it skipped the one stage the notebook exists to run and
printed the checked-in numbers as if they were its own.

You are asked to authorise Drive once. Everything lands in
`MyDrive/swarkavach_checkpoints/`, and you can delete that folder any time you
want a genuinely clean run.


In [ ]:
# Redo a stage even though Drive has its output.
# Valid names: "download", "pairs", "corpus", "train", "evaluate".
FORCE_REBUILD = []

DONE = runner.begin(force=FORCE_REBUILD)


## 1. Data

Two real corpora, both direct HTTP with no account and no approval form:

- **GramVaani** (OpenSLR 118): real Hindi *telephone* speech, transcribed, with
  gender and accent labels. This is the bonafide side of the voice branch.
- **NCSU WSPR robocalls**: 1,432 genuine illegal robocalls with transcripts,
  public domain because they are FTC evidence exhibits. Only the transcripts
  are used, so the clone is blob-filtered down to the 700 KB metadata file
  rather than the full 1.7 GB. They are the out-of-domain test in section 5:
  real fraud scripts, written by real fraudsters, that nothing here was tuned
  on. Pass `--robocall-audio` if you want the recordings as well.

Then it builds the Hindi anti-spoofing pairs. No public Hindi anti-spoofing set
exists, so each spoof renders the **same transcript** as its bonafide, and both
sides go through the same channel with the noise floor matched.

Matching the level is not enough on its own. The noise added to the spoof side
is shaped like the partner recording's own noise, because flat white noise
against real line hiss separates the two classes on spectral tilt at AUC 1.000,
and every cepstral front end here reads spectral shape directly. The detector
would score perfectly while listening to the noise rather than the voice.


In [ ]:
runner.fetch_data(pairs=600)


In [ ]:
# Confirm the set is usable BEFORE training on it. This stops the run
# rather than printing a warning: an unusable set does not fail loudly
# later, it quietly falls back to the generated corpus where both classes
# are machines, and the EER stops meaning what the report says it means.
if not runner.audit_pairs():
    raise SystemExit("anti-spoofing set is not usable, see above")


## 2. The Hinglish call corpus

Generated dialogues with gold BIO entities, gold dialogue acts and gold
per-token language tags, all emitted by the template slots rather than
annotated by hand. Rendered with real neural voices so the audio says what the
transcript says.

The `edge` backend is what makes that true. The `sim` backend produces
speech-shaped noise with the right acoustic statistics and no actual words. If
the synthesis service is unreachable the renderer falls back to `sim` per call,
so this stage now counts how many calls really came from `edge` and stops if
most did not, instead of handing you an hour of noise.


In [ ]:
runner.gen_corpus(n=480)


In [ ]:
# Listen to one. A cloned scam call, so you can check the audio says what
# the transcript says and that the caller does not sound robotic.
runner.listen("clonedxscam")


## 3. Train

Order matters here, and the splits are not casual:

- entity tagger, intent model and dialogue-act model fit on **train**
- prosody-intent anchors are recalibrated from the corpus audio
- the anti-spoofing branch fits on the **real paired set**, not the generated
  corpus, because the corpus cannot answer "person or machine" when both of
  its cells are machines
- the fusion layer fits on **dev**, never on train. It stacks on top of the
  branches, so fitting it on rows the branches trained on feeds it in-sample
  predictions that never occur at test time. The ablation arms in section 4
  are fitted on dev for the same reason, which they were not until recently.

A stage that fails now stops the run. It used to be recorded in the report and
returned normally, so a run where only fusion succeeded exited 0 and was
indistinguishable from a clean one.


In [ ]:
runner.train()


In [ ]:
# Read the anchor report. A component whose synthetic median sits clearly
# above its human median is fighting the prosody-intent feature and needs
# replacing; that happened once, to a frame-level energy measure. A gap of a
# few percent either way is "flat", and on the edge backend that is the
# expected reading: both voice cells are neural voices, and telling person
# from machine is the anti-spoofing branch's job, not this feature's.
import json
from pathlib import Path

p = Path("data/models/pim_anchors.json")
if p.exists():
    d = json.loads(p.read_text())
    print("anchors:", json.dumps(d.get("anchors"), indent=2))
    print("
component        human    synthetic   gap/span   reading")
    direction = d.get("direction") or {}
    for k, med in (d.get("medians") or {}).items():
        gap = (d.get("gap_of_span") or {}).get(k)
        if k in direction:
            reading = direction[k]
        else:  # anchors written by an older run
            reading = "ok" if (d.get("direction_ok") or {}).get(k) else "wrong"
        gap_s = f"{gap:+.3f}" if gap is not None else "   n/a"
        print(f"  {k:14s} {med.get('human'):8.4f} {med.get('synthetic'):10.4f}   "
              f"{gap_s}    {reading}")


## 4. Evaluate

Writes `data/results/*.json`, which is exactly what the local console reads.
This is the longest stage, 25 to 60 minutes, mostly the ten Gaussian mixtures
the anti-spoofing table needs.


In [ ]:
runner.evaluate(codecs="clean,g711u,g711a,gsm")


## 5. Results

The ablation table is the headline. It is reported **per cell** rather than as
one aggregate, because a single number hides the failure that matters: audio
alone should miss human scams, text alone should miss the lexically mild scams
and false-alarm on cloned benign calls, and the full arm should clear both.

A missing file is named as missing. These cells used to print nothing at all
when a result was absent, which reads as "nothing to say" rather than "this
stage never ran".


In [ ]:
runner.report()


In [ ]:
# Plots for the report. Loads the results itself rather than relying on names
# a previous cell happened to leave behind, so this works after a restart and
# says what is missing instead of raising NameError.
import matplotlib.pyplot as plt
import numpy as np

from swarkavach.config import RESULTS_DIR
from swarkavach.runner import ARM_LABEL, CELL_LABEL, load_result

ab = load_result("ablation_results.json")
asr_ = load_result("antispoof_results.json")
ttd = load_result("ttd_results.json")

have = [n for n, d in (("ablation", ab), ("antispoof", asr_), ("ttd", ttd)) if d]
if not have:
    print("nothing to plot: no results on disk yet")
else:
    print("plotting:", ", ".join(have))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    if ab:
        arms = ab.get("arms", [])
        cs = ab.get("cells", [])
        M = np.array([[ab["detection_rate"].get(a, {}).get(c) or 0 for c in cs]
                      for a in arms])
        axes[0].imshow(M, cmap="Blues", vmin=0, vmax=1)
        axes[0].set_xticks(range(len(cs)), [CELL_LABEL.get(c, c) for c in cs],
                           rotation=25, ha="right")
        axes[0].set_yticks(range(len(arms)), [ARM_LABEL.get(a, a) for a in arms])
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                axes[0].text(j, i, f"{M[i, j]:.0%}", ha="center", va="center",
                             color="white" if M[i, j] > 0.55 else "black", fontsize=9)
        axes[0].set_title("Ablation: correct decisions per cell")
    else:
        axes[0].set_title("no ablation_results.json")
        axes[0].axis("off")

    if asr_ and asr_.get("det"):
        for d in asr_["det"][:4]:
            pts = np.array(d["points"])
            axes[1].plot(pts[:, 0], pts[:, 1], label=f"{d['name']} (EER {d['eer']:.1%})")
        axes[1].plot([0, 1], [0, 1], "--", color="grey", lw=1)
        axes[1].set_xlim(0, .6)
        axes[1].set_ylim(0, .6)
        axes[1].set_xlabel("false alarm rate")
        axes[1].set_ylabel("miss rate")
        axes[1].set_title("DET curves")
        axes[1].legend(fontsize=8)
    else:
        axes[1].set_title("no DET curves")
        axes[1].axis("off")

    if ttd and ttd.get("values"):
        axes[2].hist(ttd["values"], bins=14, color="#3987e5")
        if ttd.get("median"):
            axes[2].axvline(ttd["median"], color="#d03b3b", ls="--",
                            label=f"median {ttd['median']:.1f}s")
            axes[2].legend()
        axes[2].set_xlabel("seconds heard before the alert")
        axes[2].set_ylabel("calls")
        axes[2].set_title("Time to detection")
    else:
        axes[2].set_title("no ttd_results.json")
        axes[2].axis("off")

    plt.tight_layout()
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(RESULTS_DIR / "summary.png", dpi=140, bbox_inches="tight")
    plt.show()


In [ ]:
# Everything into Drive one more time, so the zips below are a convenience
# and not the only copy. Unchanged artifacts are recognised by file count
# and total size and are not re-uploaded.
runner.finish()


## 6. Take it home

Downloads two zips. Unzip both into your local checkout, then:

```
swarkavach serve
```

and the console shows these numbers with no retraining.


In [ ]:
import json, platform, shutil, subprocess
from datetime import datetime, timezone
from pathlib import Path

# Absolute paths from config. These were relative, so they only worked
# while the working directory was still the clone.
from swarkavach.config import MODELS_DIR, RESULTS_DIR

# Stamp the provenance so the console can say which numbers came from Colab.
gpu = "cpu"
try:
    import torch
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
except Exception:
    pass

prov = RESULTS_DIR / "provenance.json"
entries = []
if prov.exists():
    try:
        entries = json.loads(prov.read_text()).get("entries", [])
    except Exception:
        entries = []
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
for p in sorted(RESULTS_DIR.glob("*.json")):
    if p.name == "provenance.json":
        continue
    entries = [e for e in entries if e.get("artifact") != p.name]
    entries.append({"artifact": p.name, "source": "colab",
                    "generated": stamp, "notes": f"Colab, {gpu}"})
prov.write_text(json.dumps({"entries": entries, "generated": stamp,
                            "host": platform.platform(), "gpu": gpu}, indent=2))

shutil.make_archive("swarkavach_results", "zip", str(RESULTS_DIR))
if MODELS_DIR.exists():
    shutil.make_archive("swarkavach_models", "zip", str(MODELS_DIR))

for f in ("swarkavach_results.zip", "swarkavach_models.zip"):
    if Path(f).exists():
        print(f, "%.1f MB" % (Path(f).stat().st_size / 1e6))

try:
    from google.colab import files
    files.download("swarkavach_results.zip")
    if Path("swarkavach_models.zip").exists():
        files.download("swarkavach_models.zip")
except Exception as exc:
    print("not on Colab, or the download was blocked:", exc)
    print("the zips are in the file browser on the left")


## 7. Push the results straight back to the repo

Optional, and nicer than downloading zips: commit the results and the trained
models to a branch, then pull them on your laptop. Needs the token to have
Contents: read and write.

It returns to `main` afterwards. Leaving the clone sitting on the results
branch made the next run's pull fail, because a branch that is ahead of main
is not a fast-forward.


In [ ]:
PUSH_BACK = True          # set False if you would rather just use the zips
RESULT_BRANCH = "colab-results"

if PUSH_BACK and TOKEN:
    import subprocess
    from datetime import datetime, timezone

    def git(*a, **kw):
        return subprocess.run(["git", *a], capture_output=True, text=True, **kw)

    git("config", "user.email", "colab@swarkavach.local")
    git("config", "user.name", "SwarKavach Colab")

    # models are gitignored by default, so force-add just the small artefacts
    git("checkout", "-B", RESULT_BRANCH)
    git("add", "-f", "data/results")
    for pat in ("data/models/*.joblib", "data/models/*.json", "data/models/*.pt"):
        git("add", "-f", pat)

    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    c = git("commit", "-m", f"Colab run {stamp}: results and trained models")
    if c.returncode != 0 and "nothing to commit" in (c.stdout + c.stderr):
        print("nothing changed")
    else:
        git("remote", "set-url", "origin", _auth_url())
        r = git("push", "-u", "origin", RESULT_BRANCH, "--force")
        git("remote", "set-url", "origin", CLEAN_URL)
        if r.returncode == 0:
            print(f"pushed to branch {RESULT_BRANCH}")
            print(f"on your laptop:  git fetch origin && git checkout {RESULT_BRANCH}")
        else:
            msg = (r.stderr or "").replace(TOKEN, "***")
            print("push failed:", msg[-600:])
            print("the zips above still work")
    # Back to main, always. Leaving the clone on the results branch made the
    # next run's clone cell fail: that branch is ahead of main, so
    # `git pull --ff-only origin main` is not a fast-forward, and the run
    # stopped before it started.
    back = git("checkout", "-f", BRANCH)
    if back.returncode == 0:
        print(f"clone returned to {BRANCH}")
    else:
        print(f"could not return to {BRANCH}, run this before the next session:")
        print(f"  !git -C {CLONE_DIR} checkout -f {BRANCH}")
elif PUSH_BACK:
    print("no token, so nothing to push. Use the zips above.")


## What to do with it

1. Unzip `swarkavach_results.zip` into `data/results/` in your local checkout.
2. Unzip `swarkavach_models.zip` into `data/models/`.
3. `swarkavach serve`, open <http://127.0.0.1:7860>, go to the Results lab.
   The provenance table shows these rows as coming from Colab.

The other notebooks in this folder go further than this one: `02` trains
RawNet on real ASVspoof data, `03` measures what a Whisper transcript costs
the entity taggers and fine-tunes MuRIL, `05` does consented voice cloning.
This notebook is the one that gets you a complete working system.
